# Pregunta 3 — CFA, SEM y comparación por género

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

## Enunciado de la pregunta 3

**3.** Como recién llegado a la consultora antes mencionada, le han incluido en un segundo proyecto, relacionado con los efectos de las condiciones de trabajo sobre la satisfacción y permanencia de los trabajadores. En este contexto, se han considerado los siguientes constructos relevantes para la investigación:

- **Actitud hacia compañeros de trabajo:** Actitud del empleado hacia/con sus compañeros de trabajo con los que interactúa regularmente.
- **Percepción del ambiente de trabajo:** Percepción del trabajador respecto a su lugar de trabajo.
- **Satisfacción con el trabajo:** Medición del nivel de satisfacción del empleado con su trabajo.
- **Compromiso organizacional:** Grado en que el empleado se identifica y siente parte de la empresa.
- **Intención de permanecer en el trabajo:** Grado en el que el trabajador pretende continuar trabajando para la empresa.
- **Características personales:** Edad, Género, y experiencia en años del trabajador.

Cada uno de estos constructos fue medido en escalas de múltiples ítems que se describen a continuación, y que fueron aplicadas en un cuestionario a 400 empleados de diversas empresas del sector de telecomunicaciones, disponibles en el archivo `Laboral.sav`.

![Tabla de constructos e ítems de la pregunta 3](../assets/p3_tabla_constructos.png)

**Actitud hacia compañeros de trabajo**

- AC1 - ¿Cuán contento está con el trabajo de sus compañeros?
- AC2 - ¿Cómo se siente en su relación respecto a sus compañeros de trabajo?
- AC3 - ¿Qué tan seguido realiza actividades con sus compañeros de trabajo en días libres?
- AC4 - En general, ¿qué tan parecidos a Ud. son sus compañeros de trabajo?

**Compromiso organizacional**

- CO1 - Mi trabajo en esta empresa me da una sensación de logros.
- CO2 - Estoy dispuesto a poner un gran esfuerzo mas allá de lo que se esperaría normalmente para ayudar a que esta empresa sea exitosa.
- CO3 - Tengo un sentido de lealtad con esta empresa.
- CO4 - Estoy orgulloso de contarle a otras personas que trabajo en esta empresa.

**Intención de permanecer en el trabajo**

- IP1 - No estoy buscando activamente otro trabajo.
- IP2 - Rara vez veo las ofertas de trabajo en sitios para esos fines.
- IP3 - No tengo interés en buscar trabajo durante los próximos 12 meses.
- IP4 - ¿Qué tan probable es que usted este trabajando para la misma empresa dentro de los próximos 12 meses?

**Percepción del ambiente de trabajo**

- PA1 - Estoy muy conforme con el ambiente físico de trabajo en esta empresa.
- PA2 - El lugar en que trabajo está diseñado para facilitar mi trabajo.
- PA3 - Hay pocos obstáculos que me hacen ser menos productivo en mi lugar de trabajo.
- PA4 - ¿Qué término describe mejor el ambiente de trabajo en su empresa?

**Satisfacción con el trabajo**

- ST1 - Considerando todas las cosas, me siento muy satisfecho cuando pienso en mi trabajo.
- ST2 - Cuando piensa en su trabajo, ¿qué tan satisfecho se siente?
- ST3 - ¿Que tan satisfecho está con su actual trabajo?
- ST4 - Como empleado, ¿qué tan satisfecho esta con su empleador?

**Características personales**

- D1 - Género.
- Edad.
- EXP - Experiencia en años.

Las escalas específicas de cada ítem se encuentran detalladas en el archivo de datos `Laboral.sav`.

Gracias a sus conocimientos en el área de comportamiento organizacional, usted ha identificado dos posibles modelos que expliquen la relación entre las condiciones de trabajo percibidas por el trabajador, y su satisfacción e intenciones de permanecer en la empresa. Estos modelos difieren principalmente en si el efecto de la precepción de ambiente de trabajo y la actitud hacia los compañeros de trabajo tienen efectos completamente mediados o parcialmente mediados sobre la intención de permanecer en la empresa.

Cada uno de los modelos se describe a continuación:

![Modelo completamente mediado](../assets/p3_modelo_completamente_mediado.png)

![Modelo parcialmente mediado](../assets/p3_modelo_parcialmente_mediado.png)

Su labor consiste en determinar:

**a)** Qué tan bueno es el modelo de medición, incluyendo medidas de confiabilidad y validez de escalas.  
**b)** Determinar cuál de los dos modelos representa de mejor forma los datos recopilados.  
**c)** Para el modelo identificado como superior, evalúe si las relaciones entre las distintas variables se comportan de la misma forma entre hombres y mujeres.  
**d)** Explique sus resultados y conclusiones.

## Respuesta

La pregunta se resuelve en dos etapas: primero se evalúa el modelo de medición mediante CFA, confiabilidad y validez; luego se comparan los modelos estructurales completamente mediado y parcialmente mediado mediante SEM. Finalmente, el modelo superior se analiza por género para evaluar si las relaciones se comportan de forma similar entre hombres y mujeres.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else Path.cwd().parent / 'data'
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)

## 3.a) Evaluación del modelo de medición

Se realiza: (a) CFA del modelo de medición; (b) confiabilidad y validez convergente/discriminante; (c) comparación de modelos estructurales anidados; y (d) comparación multigrupo por género.

Modelo completamente mediado: $PA,AC\rightarrow ST\rightarrow CO\rightarrow IP$. Modelo parcialmente mediado: agrega $PA\rightarrow IP$ y $AC\rightarrow IP$.

In [2]:
from semopy import Model, calc_stats
df,meta=pyreadstat.read_sav(DATA_DIR/'Laboral.sav',apply_value_formats=False)
constructos={'AC':['AC1','AC2','AC3','AC4'],'PA':['PA1','PA2','PA3','PA4'],'ST':['ST1','ST2','ST3','ST4'],'CO':['CO1','CO2','CO3','CO4'],'IP':['IP1','IP2','IP3','IP4']}
obs=sum(constructos.values(),[]); datos=df[obs+['GENERO']].dropna()
print('Casos:',len(datos),'Hombres:',sum(datos.GENERO==0),'Mujeres:',sum(datos.GENERO==1))

Casos: 400 Hombres: 200 Mujeres: 200


## 3.b) Comparación de los modelos estructurales

Criterios orientativos: CFI/TLI ≥0.90 (preferible ≥0.95), RMSEA ≤0.08 (preferible ≤0.06). Para modelos anidados se utiliza $\Delta\chi^2$ con grados de libertad iguales a la diferencia entre modelos.

In [3]:
med='\n'.join(f"{k} =~ {' + '.join(v)}" for k,v in constructos.items())
completo=med+'\nST ~ PA + AC\nCO ~ ST\nIP ~ CO'
parcial=med+'\nST ~ PA + AC\nCO ~ ST\nIP ~ CO + PA + AC'
def ajustar(desc,d):
 m=Model(desc); m.fit(d); return m,calc_stats(m).iloc[0],m.inspect(std_est=True)
mc,fc,pc=ajustar(completo,datos); mp,fp,pp=ajustar(parcial,datos)
indices=['chi2','DoF','CFI','TLI','RMSEA','AIC','BIC']
print(pd.DataFrame({'Completo':fc[indices],'Parcial':fp[indices]}).round(4).to_string())
dchi=fc.chi2-fp.chi2; ddf=fc.DoF-fp.DoF; print(f'\nDiferencia chi-cuadrado={dchi:.4f}, gl={ddf:.0f}, p={stats.chi2.sf(dchi,ddf):.4g}')

       Completo   Parcial
chi2   373.6224  311.1933
DoF    165.0000  163.0000
CFI      0.9480    0.9631
TLI      0.9401    0.9570
RMSEA    0.0563    0.0477
AIC     88.1319   92.4440
BIC    267.7478  280.0429

Diferencia chi-cuadrado=62.4291, gl=2, p=2.778e-14


## 3. Confiabilidad y validez

Se reportan alfa, confiabilidad compuesta (CR) y AVE. Se esperan alfa/CR ≥0.70 y AVE ≥0.50. Para discriminante se usa Fornell–Larcker y HTMT (<0.85).

In [4]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
filas=[]
for c,its in constructos.items():
 r=pp[(pp.op=='~')&(pp.rval==c)&pp.lval.isin(its)].set_index('lval')['Est. Std'].reindex(its).astype(float)
 cr=r.sum()**2/(r.sum()**2+(1-r**2).sum()); ave=(r**2).mean()
 filas.append([c,alpha(datos[its]),cr,ave,r.min(),r.max()])
calidad=pd.DataFrame(filas,columns=['Constructo','Alfa','CR','AVE','Carga mínima','Carga máxima'])
print(calidad.round(4).to_string(index=False))
scores=mp.predict_factors(datos); print('\nCorrelaciones latentes:'); print(scores.corr().round(3).to_string())
def htmt(a,b):
 A,B=constructos[a],constructos[b]; R=datos[A+B].corr().abs(); h=R.loc[A,B].values.mean(); ma=R.loc[A,A].values[np.triu_indices(4,1)].mean(); mb=R.loc[B,B].values[np.triu_indices(4,1)].mean(); return h/np.sqrt(ma*mb)
print('\nHTMT:')
for i,a in enumerate(constructos):
 for b in list(constructos)[i+1:]: print(f'{a}-{b}: {htmt(a,b):.3f}')

Constructo   Alfa     CR    AVE  Carga mínima  Carga máxima
        AC 0.8908 0.8941 0.6786        0.8140        0.8360
        PA 0.8474 0.8577 0.6020        0.6956        0.8245
        ST 0.8112 0.8106 0.5171        0.6964        0.7366
        CO 0.8227 0.8345 0.5642        0.5902        0.8801
        IP 0.8863 0.8784 0.6445        0.7211        0.8517

Correlaciones latentes:
       AC     CO     IP     PA     ST
AC  1.000  0.280  0.333  0.288  0.048
CO  0.280  1.000  0.587  0.449  0.251
IP  0.333  0.587  1.000  0.607  0.246
PA  0.288  0.449  0.607  1.000  0.293
ST  0.048  0.251  0.246  0.293  1.000

HTMT:
AC-PA: 0.260
AC-ST: 0.044
AC-CO: 0.275
AC-IP: 0.310
PA-ST: 0.231
PA-CO: 0.495
PA-IP: 0.569
ST-CO: 0.193
ST-IP: 0.217
CO-IP: 0.501


## 4. Relaciones estructurales del modelo superior

In [5]:
paths=pp[(pp.op=='~')&pp.lval.isin(['ST','CO','IP'])][['lval','rval','Estimate','Est. Std','Std. Err','z-value','p-value']]
print(paths.round(4).to_string(index=False))

lval rval  Estimate  Est. Std  Std. Err   z-value   p-value
  ST   PA    0.2015    0.2601  0.049188  4.095605  0.000042
  ST   AC   -0.0229   -0.0266  0.051471 -0.444425  0.656735
  CO   ST    0.3279    0.2170  0.092422  3.547838  0.000388
  IP   CO    0.1677    0.3747  0.025445  6.591264       0.0
  IP   PA    0.2100    0.4010  0.029845  7.036789       0.0
  IP   AC    0.0718    0.1234  0.029166  2.461081  0.013852


## 3.c) Comparación de relaciones entre hombres y mujeres

Se estima el mismo modelo parcial por separado y se compara cada coeficiente no estandarizado con
$$z=(b_H-b_M)/\sqrt{SE_H^2+SE_M^2}.$$
Esta prueba evalúa diferencias de relaciones estructurales. Para una aplicación estrictamente confirmatoria futura sería recomendable complementar con invariancia configural, métrica y escalar.

In [6]:
gr={}
for g,nombre in [(0,'Hombres'),(1,'Mujeres')]:
 m,f,p=ajustar(parcial,datos[datos.GENERO==g]); gr[g]=p[(p.op=='~')&p.lval.isin(['ST','CO','IP'])]
 print(nombre,'n=',sum(datos.GENERO==g),'CFI=',round(f.CFI,3),'RMSEA=',round(f.RMSEA,3))
fil=[]
for lhs,rhs in [('ST','PA'),('ST','AC'),('CO','ST'),('IP','CO'),('IP','PA'),('IP','AC')]:
 a=gr[0][(gr[0].lval==lhs)&(gr[0].rval==rhs)].iloc[0]; b=gr[1][(gr[1].lval==lhs)&(gr[1].rval==rhs)].iloc[0]
 z=(a.Estimate-b.Estimate)/np.sqrt(float(a['Std. Err'])**2+float(b['Std. Err'])**2)
 fil.append([f'{rhs}→{lhs}',a.Estimate,b.Estimate,z,2*stats.norm.sf(abs(z))])
print(pd.DataFrame(fil,columns=['Relación','Hombres','Mujeres','z diferencia','p']).round(4).to_string(index=False))

Hombres n= 200 CFI= 0.945 RMSEA= 0.052
Mujeres n= 200 CFI= 0.944 RMSEA= 0.058
Relación  Hombres  Mujeres  z diferencia      p
   PA→ST   0.2757   0.1624        1.0518 0.2929
   AC→ST  -0.2050   0.3830       -3.4729 0.0005
   ST→CO   0.1797   0.6035       -2.1268 0.0334
   CO→IP   0.1379   0.2066       -1.3755 0.1690
   PA→IP   0.1220   0.2507       -2.1257 0.0335
   AC→IP  -0.0173   0.0322       -0.5043 0.6140


## 3.d) Explicación y conclusión

Todas las escalas muestran confiabilidad adecuada (alfa 0.811–0.891; CR 0.811–0.894), AVE superior a 0.50 y discriminación adecuada (HTMT máximo 0.569). El modelo parcial presenta mejor ajuste (CFI=0.963, TLI=0.957, RMSEA=0.048) que el completo (CFI=0.948, RMSEA=0.056), con mejora significativa, $\Delta\chi^2(2)=62.43$, $p<0.001$. Por ello se selecciona el **modelo parcialmente mediado**.

En el modelo parcial, PA→ST, ST→CO, CO→IP, PA→IP y AC→IP son significativos; AC→ST no lo es. Por género, difieren AC→ST ($p<0.001$), ST→CO ($p=0.033$) y PA→IP ($p=0.034$); las demás relaciones no difieren al 5%. Por tanto, **el modelo no se comporta completamente igual entre hombres y mujeres**.